<!-- bootcamp-header: generated by tools/build_headers.py, edit the README timetable instead -->
# pandas

**Session:** Wednesday 7 October 2026, 10:30-12:30, room S.R.118  
**Tutors:** Loren Verreyen / Caroline Vandyck  
**Exercises:** [`13_EX_Pandas.ipynb`](https://github.com/mikekestemont/dtaantwerp26-27.github.io/blob/DTA_Bootcamp_2026_students/exercises/questions/13_EX_Pandas.ipynb) (solutions: [`13_SOL_Pandas.ipynb`](https://github.com/mikekestemont/dtaantwerp26-27.github.io/blob/DTA_Bootcamp_2026_students/exercises/solutions/13_SOL_Pandas.ipynb))  

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mikekestemont/dtaantwerp26-27.github.io/blob/DTA_Bootcamp_2026_students/notebooks/13_W3_Wed_Pandas.ipynb)

In [ ]:
# Run this cell only if you are working on Google Colab: it downloads the course
# material (notebooks and data) so that the file paths in this notebook work.
# On your own computer you can skip it.
!git clone --quiet --depth 1 --branch DTA_Bootcamp_2026_students https://github.com/mikekestemont/dtaantwerp26-27.github.io.git bootcamp
%cd bootcamp/notebooks

## Tables with `pandas`

Much of the data you will meet is not running text but a **table**: a spreadsheet of metadata, a CSV file of annotations, the output of a corpus tool. Python's standard tool for tables is the library **`pandas`**, and this session is a first, practical tour of it, on one dataset. You will not learn all of `pandas` in two hours, nobody does; you will learn the dozen operations that cover most of what a text analyst does with a table, and how to look up the rest.

After this session you can:

- load a CSV file into a `DataFrame` and inspect it;
- select columns and rows, and filter rows with a condition;
- count and summarise a column, and compare groups with `groupby()`;
- add a column computed from other columns;
- make a bar chart of a result, and save a table back to CSV.

## A first look

`pandas` is imported under the alias `pd` by universal convention. Its central object is the **`DataFrame`**: a table with named columns and numbered rows. The usual way to get one is to read a CSV file (*comma-separated values*: one row per line, values separated by commas, column names on the first line). Our dataset is the passenger list of the *Titanic*, a classic for learning `pandas` because every column is easy to understand.

In [ ]:
import pandas as pd

df = pd.read_csv('../data/titanic.csv')
df.head()

`df` is the conventional name for a DataFrame. `.head()` shows the first five rows (`.tail()` the last five; give a number to see more). The bold numbers on the left are the **index**: the row labels, here just 0, 1, 2, ... Every column has a name and a type, and a first inspection is always the same four questions: how big, which columns, which types, anything missing?

In [ ]:
print(df.shape)         # (rows, columns)
print(len(df))          # rows
print(df.columns)

In [ ]:
df.info()

`.info()` answers all four at once. The types: `int64` and `float64` are numbers, `object` is text (or mixed). The *non-null* counts show that `Age` is missing for 177 passengers and `Cabin` for most; missing values are shown as `NaN` (*not a number*) in the table, and `pandas` skips them when it computes averages.

`.describe()` gives the standard statistics of every numerical column in one go:

In [ ]:
df.describe()

A CSV file is not always comma-separated, and not always UTF-8: `pd.read_csv(path, sep=';', encoding='latin-1')` reads a semicolon-separated file in another encoding. If a file loads as one column, or with strange characters, these two arguments are the first thing to check.

## Columns and rows

A single column, selected with its name between square brackets, is a **`Series`**: one-dimensional, with the same index as the table. Most of what you do in `pandas` you do to a Series.

In [ ]:
ages = df['Age']
print(type(ages))
ages.head()

In [ ]:
print(ages.mean(), ages.min(), ages.max())
print(ages.median())

Several columns at once: a *list* of names inside the brackets (hence the double brackets). The result is again a DataFrame.

In [ ]:
df[['Name', 'Age', 'Survived']].head()

Rows are selected by position with `.iloc[]` (like a list) or by index label with `.loc[]`; with the default index both give the same here. `.loc[]` also takes a column name as a second argument, which is the way to read one cell:

In [ ]:
print(df.iloc[0])            # the first row, as a Series
print(df.loc[0, 'Name'])     # one cell: row 0, column Name
df.iloc[10:15]               # rows 10 to 14

## Filtering rows with a condition

The operation you will use most. A comparison on a column gives a Series of `True`/`False`, one per row, a **boolean mask**; putting that mask between the brackets of the DataFrame keeps the rows where it is `True`:

In [ ]:
df['Age'] > 60

In [ ]:
elderly = df[df['Age'] > 60]
print(len(elderly))
elderly[['Name', 'Age', 'Survived']]

Read `df[df['Age'] > 60]` as "the rows of `df` where `Age` is above 60". Text columns compare with `==`, and a Series of strings has string methods behind `.str`, such as `.str.contains()` (which, like `re.findall()`, reads its argument as a pattern):

In [ ]:
women = df[df['Sex'] == 'female']
print(len(women))

boys = df[df['Name'].str.contains('Master')]      # 'Master' was the title of a boy
boys[['Name', 'Age']].head()

To combine conditions you need `&` (and) and `|` (or), **not** `and` and `or`, and every condition goes between **round brackets**. This is the place where `&` and `|` finally belong; they work element by element on the two masks, which is what `and` and `or` cannot do.

In [ ]:
df[(df['Sex'] == 'female') & (df['Pclass'] == 1)].head()

In [ ]:
df[(df['Age'] < 10) | (df['Age'] > 70)][['Name', 'Age']]

`.isin()` tests membership in a list, which is shorter than several `|`:

In [ ]:
df[df['Embarked'].isin(['C', 'Q'])].shape

### Class exercises

1. How many passengers were younger than 18? How many of those were in third class? Show the names and ages of the passengers older than 70 who survived.

In [ ]:
# your code here

## Counting and summarising

`.value_counts()` counts how often each value occurs in a column, most frequent first: it is `Counter` for a Series. `.unique()` lists the distinct values and `.nunique()` counts them.

In [ ]:
print(df['Pclass'].value_counts())
print(df['Embarked'].unique(), df['Embarked'].nunique())

`.sort_values()` sorts a Series or, with a column name, a whole DataFrame. `.idxmax()` gives the index label of the largest value, which combined with `.loc[]` answers "who?" questions:

In [ ]:
df.sort_values('Fare', ascending=False)[['Name', 'Fare']].head()

In [ ]:
oldest = df['Age'].idxmax()
print(df.loc[oldest, 'Name'], df.loc[oldest, 'Age'])

A trick you will use constantly: the `Survived` column holds 0 and 1, so its **mean is the survival rate**. The same works for any yes/no column, and for any mask, because `True` counts as 1:

In [ ]:
print(df['Survived'].mean())
print((df['Age'] > 60).mean())      # the share of passengers over 60

## Comparing groups: `groupby()`

The question behind most tables is "does this differ between groups?" `groupby()` splits the table by the values of one column, and a summary method after it is applied to each group separately. Survival rate by sex:

In [ ]:
df.groupby('Sex')['Survived'].mean()

Read it left to right: *group the rows by `Sex`, take the `Survived` column, average it per group*. Any summary works after the column: `.mean()`, `.sum()`, `.count()`, `.max()`, and `.size()` for the number of rows in each group.

In [ ]:
print(df.groupby('Pclass')['Fare'].mean())
print(df.groupby('Pclass').size())

Two grouping columns give a group for every combination, as a list of two names:

In [ ]:
df.groupby(['Pclass', 'Sex'])['Survived'].mean()

### Class exercise

2. What was the average age per class? And the survival rate per port of embarkation (`Embarked`)? Which class had the most children under 10 (group a filtered table)?

In [ ]:
# your code here

## New columns

A new column is made by assigning to a name that does not exist yet, and the value can be computed from other columns, row by row, without a loop: arithmetic, comparisons and `.str` methods all work on whole columns at once.

In [ ]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1      # siblings/spouses + parents/children + the passenger
df['IsChild'] = df['Age'] < 12
df['Surname'] = df['Name'].str.split(',').str[0]

df[['Name', 'Surname', 'FamilySize', 'IsChild']].head()

When the computation needs a function of your own, `.apply()` calls it on every value of a column and collects the results. Here we turn the class number into a word:

In [ ]:
def class_name(number):
    return {1: 'first', 2: 'second', 3: 'third'}[number]

df['ClassName'] = df['Pclass'].apply(class_name)
df['ClassName'].value_counts()

Two more things about missing values: `.isna()` marks them (and `.isna().sum()` counts them per column), and `.dropna()` drops the rows that have any. Do the second only when you mean it: `df.dropna()` on this table would throw away most passengers because of the `Cabin` column, so name the columns that matter.

In [ ]:
print(df.isna().sum())
with_age = df.dropna(subset=['Age'])
print(len(df), len(with_age))

## A plot

A Series has a `.plot()` method, which draws it with the library `matplotlib`. `kind='bar'` gives a bar chart, the right choice for a result of `value_counts()` or `groupby()`. `matplotlib` itself is imported for the title and labels, and `plt.show()` displays the figure.

In [ ]:
import matplotlib.pyplot as plt

df.groupby('Pclass')['Survived'].mean().plot(kind='bar')
plt.title('Survival rate by class')
plt.xlabel('class')
plt.ylabel('share survived')
plt.show()

In [ ]:
df['Age'].plot(kind='hist', bins=20)
plt.title('Age of the passengers')
plt.show()

`kind='barh'` gives horizontal bars, which suit long labels; `kind='hist'` a histogram of a numerical column. That is as far as plotting goes in this course; the module on data visualisation takes it from here.

## Saving

`.to_csv()` writes a DataFrame back to a file. `index=False` leaves out the row numbers, which you rarely want in the file.

In [ ]:
survivors = df[df['Survived'] == 1][['Name', 'Sex', 'Age', 'Pclass']]
survivors.to_csv('survivors.csv', index=False)

pd.read_csv('survivors.csv').head(3)

## Making a table yourself

Tables do not only come from files. A list of dictionaries, one per row with the column names as keys (the shape you met in the session on dictionaries), turns into a DataFrame directly, and so does a `Counter` via its `.items()`:

In [ ]:
from collections import Counter

rows = [{'title': 'Emma', 'author': 'Austen', 'year': 1815},
        {'title': 'Dracula', 'author': 'Stoker', 'year': 1897},
        {'title': 'Middlemarch', 'author': 'Eliot', 'year': 1871}]
novels = pd.DataFrame(rows)
print(novels)

counts = Counter('the cat sat on the mat with the hat'.split())
frequencies = pd.DataFrame(counts.items(), columns=['word', 'count']).sort_values('count', ascending=False)
print(frequencies)

### Class exercise

3. Read `alice.txt`, tokenise it with `re.findall(r'[a-z]+', text.lower())`, count the tokens with a `Counter`, turn the counter into a DataFrame with the columns `word` and `count`, add a column `length` with the length of each word, and show the ten most frequent words of more than six letters.

In [ ]:
import re

# your code here

## Common mistakes

Four things that go wrong in everyone's first week with `pandas`.

In [ ]:
# This cell produces an error on purpose: 'and' does not work on columns; use & with round brackets
df[df['Sex'] == 'female' and df['Pclass'] == 1]

In [ ]:
# This cell produces an error on purpose: & without brackets around each condition
df[df['Sex'] == 'female' & df['Pclass'] == 1]

In [ ]:
# This cell produces an error on purpose: a column name that does not exist (case matters)
df['age'].mean()

In [ ]:
# Not an error, but a silent trap: a method returns a new table; the original is unchanged unless you assign
df.sort_values('Age')
print(df['Age'].head(3))          # not sorted
df_sorted = df.sort_values('Age')
print(df_sorted['Age'].head(3))   # sorted

## Quick reference

| Task | Code |
| --- | --- |
| load, inspect | `pd.read_csv(path)`, `.head()`, `.shape`, `.columns`, `.info()`, `.describe()` |
| one column, several | `df['col']`, `df[['a', 'b']]` |
| rows | `df.iloc[i]`, `df.loc[i, 'col']`, `df[mask]` |
| conditions | `df['col'] > 5`, `==`, `.str.contains()`, `.isin([...])`, `(a) & (b)`, `(a) \| (b)` |
| summarise | `.mean()`, `.sum()`, `.min()`, `.max()`, `.median()`, `.value_counts()`, `.unique()`, `.nunique()` |
| sort, locate | `.sort_values('col', ascending=False)`, `.idxmax()` |
| groups | `df.groupby('col')['other'].mean()`, `.groupby(['a', 'b'])` |
| new column | `df['new'] = df['a'] + df['b']`, `df['col'].apply(function)`, `.str.` methods |
| missing values | `.isna().sum()`, `.dropna(subset=['col'])` |
| plot, save | `.plot(kind='bar')`, `plt.title()`, `plt.show()`, `.to_csv(path, index=False)` |

## References

- [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html): the official quick tour
- [pandas cheat sheet](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf) (PDF)
- [The Titanic dataset](https://www.kaggle.com/c/titanic/data): what the columns mean